# PettingZoo agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/pettingzoo/agent_baseline.ipynb)

A minimal, runnable **multi-agent** baseline for any ml-arena PettingZoo competition
(Connect-Four, Chess, Texas Hold'em, the PettingZoo Atari suite, ...). It plays a
**random *legal* move** — it reads the `action_mask` and only picks allowed actions.

> Open in Colab, run top to bottom. Edit your API token and `COMPETITION_ID`
> (default `65` — **Connect Four**).

## 0. Setup

In [9]:
import os
import subprocess
import sys

from dotenv import load_dotenv

load_dotenv()  # charge MLARENA_API_TOKEN depuis .env en local

# Sur Colab (ou kernel distant) : installer si besoin. En local avec uv, les deps sont déjà là.
try:
    import mlarena
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlarena-sdk==0.3.0", "numpy", "python-dotenv"])
    import mlarena

# Token : mets-le dans .env (MLARENA_API_TOKEN=...), ou colle-le ici (ne commit pas ton vrai token).
API_TOKEN = os.environ.get("MLARENA_API_TOKEN", "mlk_xxx")  # Profile -> API Keys
COMPETITION_ID = 65

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`** to disk with `%%writefile`. We upload the whole file
(so its `import`s come along) — that is what the worker runs. Keep it **self-contained**:
it is executed on its own, so it may not import the environment or other notebook cells,
only public packages available in the runtime.

Same contract as single-agent, plus two multi-agent details:

- `reset(env_player_name, episode_index)` is called at the start of each episode — the
  executor may seat you as a different player each game; remember your role here.
- For turn-based board games the observation carries an **`action_mask`** (a 0/1 vector
  of legal actions). You **must** pick an action where the mask is `1`, or you forfeit.

In [10]:
%%writefile agent.py
"""
PettingZoo Agent Template (multi-agent RL) — flex_v1 contract.

flexkit's AEC loop drives your agent:
  1. Agent()                                   zero-arg constructor
  2. setup(observation_space, action_space)    once, before the first episode
     (the spaces arrive dict-encoded — decode with flexkit.spaces.decode_space)
  3. reset(env_player_name, episode_index)     at the start of EVERY episode
     (REQUIRED — per-episode role rotation may reassign you to another slot)
  4. choose_action(observation, reward, terminated, truncated, info, action_mask)
     once per turn; honor action_mask when present; return None once done
"""
import random


class Agent:
    def __init__(self):
        self.observation_space = None
        self.action_space = None
        self.env_player_name = ""
        self.episode_index = 0

    def setup(self, observation_space, action_space):
        # flexkit is provided by the platform runtime (not needed to run this
        # notebook locally); import it here so `import agent` works anywhere.
        from flexkit.spaces import decode_space
        self.observation_space = decode_space(observation_space)
        self.action_space = decode_space(action_space)
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        if action_mask is not None:
            legal = [i for i, ok in enumerate(action_mask) if ok]
            if legal:
                # TODO: replace random choice with your policy over legal moves.
                return random.choice(legal)
        return self.action_space.sample()


Overwriting agent.py


## 2. (optional) Sanity-check it locally

Fake a Connect-Four action mask (7 columns, one full) and confirm we only play legal.

In [11]:
# Local smoke test — on the platform, flexkit calls setup() with the real specs.
from agent import Agent


class _FakeDiscrete:
    def __init__(self, n):
        self.n = n

    def sample(self):
        import random
        return random.randrange(self.n)


agent = Agent()
agent.action_space = _FakeDiscrete(7)   # setup() does this for you on the platform
agent.reset("player_0", 0)
mask = [1, 1, 0, 1, 1, 1, 1]   # column 2 is full
print("legal action:", agent.choose_action(observation=None, action_mask=mask))


legal action: 0


## 3. Submit

In [ ]:
# Uploads agent.py (with its imports intact), then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, files=["agent.py"])
print(result)

# Stream status until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

{'attache_agent_id': 8537, 'agent_id': 8537, 'deploy': {'deployment_id': 8319, 'message': 'Deployment started successfully', 'status': 'deploy_queue'}}
[deploy_queue] Deployment queued successfully
  queued: position=1/2 waiting=0s
  queued: position=1/2 waiting=5s
  queued: position=1/2 waiting=10s
  queued: position=1/2 waiting=15s
  queued: position=1/2 waiting=20s
  queued: position=1/2 waiting=25s
  queued: position=1/2 waiting=30s
  queued: position=1/2 waiting=36s
  queued: position=1/2 waiting=41s
  queued: position=1/2 waiting=46s
  queued: position=1/2 waiting=51s
  queued: position=1/2 waiting=56s
  queued: position=1/2 waiting=61s
  queued: position=1/2 waiting=66s
  queued: position=1/2 waiting=71s
  queued: position=1/2 waiting=76s
  queued: position=1/2 waiting=81s
  queued: position=1/2 waiting=86s
  queued: position=1/2 waiting=92s
  queued: position=1/2 waiting=97s
  queued: position=1/2 waiting=102s
  queued: position=1/2 waiting=107s
  queued: position=1/2 waiting=1

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

- **Search / plan:** for perfect-information games (Connect-Four, Chess) a few plies of
  minimax or MCTS over the legal moves already crushes the random baseline.
- **Self-play RL:** train a policy/value net (AlphaZero-style) offline, upload the
  weights next to `agent.py`.
- Always filter by `action_mask` — an illegal move loses the game instantly.
- **Reuse this notebook** for any PettingZoo competition; only `COMPETITION_ID` changes.